# BERT / MNLI regression analysis

Read completed or snapshotted TraceML runs copied into `logs/`. No GPU,
network connection or running aggregator is needed. Set the paths below;
one run is enough to inspect, two runs allow a paired comparison.

The notebook reads SQLite **read-only**. Timing derivation and clock selection
use the existing TraceML analyzer. No missing value is replaced with zero.
This experiment has no measured result until you run it on the GPU machine.


In [ ]:
import json
import sqlite3
from contextlib import closing
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from traceml_ai.step_time.analysis import StepTimeAnalyzer
from traceml_ai.step_time.model import StepTimeLoadRequest
from traceml_ai.step_time.sqlite import SQLiteStepTimeRepository

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "src/traceml_ai").is_dir()
)
RUNS = {
    "affected": ROOT / "logs/mnli-affected-pilot-1",
    "fixed": ROOT / "logs/mnli-fixed-pilot-1",
}
SKIP_UPDATES = 20  # Exclude the first 20 updates from averages, not raw plots.
BIN_STEPS = 50  # Means per step bin; raw points remain visible.
X_AXIS = "step"  # Or "elapsed_min" when source clocks are synchronized.

## Load and check coverage

Prefer `aggregator/telemetry.snapshot.sqlite`, created with the README's SQLite
backup command. Otherwise read the local `aggregator/telemetry` file. Reading
the current database does not recover data already deleted by retention.

`common_steps` are steps present on every **observed** rank. Check the expected
rank count and Trainer update count too: common alignment cannot reveal an
entirely absent rank on its own. A mid-run snapshot or interrupted run can
have partial coverage and may lack final Trainer files.


In [ ]:
def read_run(label, run_dir):
    database = run_dir / "aggregator/telemetry.snapshot.sqlite"
    if not database.is_file():
        database = run_dir / "aggregator/telemetry"
    if not database.is_file():
        raise FileNotFoundError(
            f"Copy the SQLite snapshot into {run_dir / 'aggregator'}"
        )

    def read_json(relative):
        path = run_dir / relative
        return json.loads(path.read_text()) if path.is_file() else {}

    experiment = read_json("training/experiment.json")
    state = read_json("training/trainer_state.json")
    manifest = read_json("manifest.json")
    expected = experiment.get("world_size")
    if expected is None:
        launch = manifest.get("launch", {})
        if "nnodes" in launch and "nproc_per_node" in launch:
            expected = launch["nnodes"] * launch["nproc_per_node"]

    with closing(
        sqlite3.connect(database.resolve().as_uri() + "?mode=ro", uri=True)
    ) as conn:
        conn.execute("PRAGMA query_only=ON")
        conn.execute(
            "BEGIN"
        )  # All families come from the same SQLite snapshot.
        tables = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
            conn,
        )["name"].tolist()
        snapshot = SQLiteStepTimeRepository(conn).load_summary(
            StepTimeLoadRequest()
        )
        window = StepTimeAnalyzer().analyze(snapshot, window_size=None)
        timestamps = dict(
            conn.execute(
                "SELECT id, timestamp_unix_ns FROM step_time_samples"
            ).fetchall()
        )
        memory = (
            pd.read_sql_query(
                """
            WITH latest AS (
                SELECT *, ROW_NUMBER() OVER (
                    PARTITION BY global_rank, step, device ORDER BY id DESC
                ) AS occurrence
                FROM step_memory_samples
                WHERE global_rank IS NOT NULL AND step IS NOT NULL
            )
            SELECT global_rank AS rank, step, device, timestamp_unix_ns,
                   peak_alloc_bytes, peak_reserved_bytes
            FROM latest WHERE occurrence=1 ORDER BY rank, step
        """,
                conn,
            )
            if "step_memory_samples" in tables
            else pd.DataFrame()
        )

    source_times = {
        (row.global_rank, row.step): timestamps[row.source_id]
        for row in snapshot.rows
    }
    records = [
        {
            "rank": rank.global_rank,
            "step": step.step,
            "timestamp_unix_ns": source_times[(rank.global_rank, step.step)],
            **asdict(step.values),
        }
        for rank in window.rank_facts
        for step in rank.steps
    ]
    timing = pd.DataFrame(records)
    if timing.empty:
        raise ValueError(f"No common completed timing steps in {database}")
    timing["timestamp_unix_ns"] = pd.to_numeric(
        timing["timestamp_unix_ns"], errors="coerce"
    )
    origin = timing["timestamp_unix_ns"].min()
    timing["elapsed_min"] = (timing["timestamp_unix_ns"] - origin) / 60e9
    if not memory.empty:
        memory["elapsed_min"] = (memory["timestamp_unix_ns"] - origin) / 60e9

    ranks = {row.global_rank for row in snapshot.rows}
    coverage = {
        "run": label,
        "expected_ranks": expected,
        "observed_ranks": len(ranks),
        "common_steps": len(window.steps),
        "trainer_updates": state.get("global_step"),
        "first_step": min(window.steps),
        "last_step": max(window.steps),
        "clock": window.clock,
        "memory_rows": len(memory),
    }
    if expected is None or ranks != set(range(expected)):
        print(
            f"{label}: rank coverage requires inspection: expected={expected}, ranks={sorted(ranks)}"
        )
    if state.get("global_step") != len(window.steps):
        print(
            f"{label}: retained common steps differ from final Trainer updates (or state is absent)."
        )
    if experiment.get("variant") not in (None, label):
        raise ValueError(
            f"{label}: experiment.json labels this run {experiment['variant']!r}"
        )

    destination = run_dir / "analysis"
    destination.mkdir(exist_ok=True)
    timing.to_csv(destination / "step_time.csv", index=False)
    memory.to_csv(destination / "step_memory.csv", index=False)
    pd.Series(coverage).to_json(destination / "coverage.json", indent=2)
    return {
        "path": run_dir,
        "database": database,
        "tables": tables,
        "experiment": experiment,
        "state": state,
        "coverage": coverage,
        "timing": timing,
        "memory": memory,
        "train_metrics": read_json("training/train_results.json"),
    }


runs = {}
for label, run_dir in RUNS.items():
    if run_dir.is_dir():
        runs[label] = read_run(label, run_dir)
    else:
        print(f"Not loaded: {label} ({run_dir})")
if not runs:
    raise FileNotFoundError(
        "Copy a GPU run locally, then set RUNS in the first code cell."
    )
display(pd.DataFrame([run["coverage"] for run in runs.values()]))

## Check the pair before interpreting a difference

The two runs should differ in Transformers commit and label only. Verify the
same data/subset fingerprints, settings, world size and runtime environment.
Different timing clocks or partial coverage make phase averages unsuitable
for a direct comparison. Seeding improves repeatability; it does not promise
bitwise-identical CUDA training.


In [ ]:
if len(runs) == 2:
    left, right = [run["experiment"] for run in runs.values()]
    fields = (
        "model",
        "dataset",
        "fingerprints",
        "samples",
        "world_size",
        "runtime",
    )
    differences = [key for key in fields if left.get(key) != right.get(key)]
    for key in (
        "train_fraction",
        "epochs",
        "batch_size",
        "max_length",
        "num_workers",
        "seed",
        "report_to",
    ):
        if left.get("arguments", {}).get(key) != right.get(
            "arguments", {}
        ).get(key):
            differences.append(key)
    if differences:
        print("Check configuration differences:", differences)
    else:
        print("Recorded workload and runtime settings match.")
    if len({run["coverage"]["clock"] for run in runs.values()}) != 1:
        print(
            "Timing clocks differ; do not compare phase averages across these runs."
        )

means = []
for label, run in runs.items():
    # Fresh runs only: the lowest retained step is not necessarily the first
    # training update when retention or missing telemetry removed the prefix.
    frame = run["timing"].loc[run["timing"]["step"] >= SKIP_UPDATES]
    columns = [
        "step_time_ms",
        "input_wait_ms",
        "compute_ms",
        "dataloader_fetch_cpu_ms",
    ]
    rank_means = frame.groupby("rank")[columns].mean()
    rank_means["samples"] = frame.groupby("rank").size()
    rank_means.insert(0, "run", label)
    rank_means.insert(1, "clock", run["coverage"]["clock"])
    means.append(rank_means.reset_index())
display(pd.concat(means, ignore_index=True))

## Where did time go?

Faint points are raw aligned steps, including startup and outliers. Lines are
means in `BIN_STEPS`-wide step bins. These plots use the analyzer's selected
clock. DataLoader fetch is supplemental CPU timing; it is not added to Step
Time. Backward can include DDP synchronization.

The step axis avoids cross-host clock offsets. Use `elapsed_min` only with
synchronized clocks. Epoch-end validation occurs between training steps and
is excluded from TraceML's measured step duration.


In [ ]:
for label, run in runs.items():
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    for axis, metric in zip(
        axes, ("step_time_ms", "input_wait_ms", "compute_ms")
    ):
        for rank, group in run["timing"].groupby("rank"):
            axis.scatter(group[X_AXIS], group[metric], s=3, alpha=0.12)
            binned = group.groupby(group["step"] // BIN_STEPS)[
                [X_AXIS, metric]
            ].mean()
            axis.plot(binned[X_AXIS], binned[metric], label=f"rank {rank}")
        axis.set_ylabel(metric.replace("_ms", "") + " (ms)")
        axis.grid(alpha=0.2)
    axes[0].legend(ncol=4)
    axes[-1].set_xlabel(
        "Optimizer update (TraceML step ID)"
        if X_AXIS == "step"
        else "Elapsed minutes"
    )
    fig.suptitle(
        f"{label}: training timing ({run['coverage']['clock']} clock)"
    )
    fig.tight_layout()
    fig.savefig(run["path"] / "analysis/timing.png", dpi=150)
    plt.show()

## GPU allocator memory

Per-step peaks describe the training process allocator, not all memory used
on the GPU. Reserved memory above allocated memory is normal caching and does
not establish a leak. These are raw retained memory rows, not restricted to
the common timing steps. Gaps remain missing observations.


In [ ]:
for label, run in runs.items():
    if run["memory"].empty:
        print(f"{label}: no step-memory samples were saved.")
        continue
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    for axis, metric in zip(axes, ("peak_alloc_bytes", "peak_reserved_bytes")):
        for (rank, device), group in run["memory"].groupby(["rank", "device"]):
            axis.scatter(
                group[X_AXIS],
                group[metric] / 1024**3,
                s=3,
                alpha=0.35,
                label=f"rank {rank}, {device}",
            )
        axis.set_ylabel(metric.replace("_bytes", "") + " (GiB)")
        axis.grid(alpha=0.2)
    axes[0].legend(ncol=2)
    axes[-1].set_xlabel(X_AXIS)
    fig.suptitle(f"{label}: per-step allocator peaks")
    fig.tight_layout()
    fig.savefig(run["path"] / "analysis/memory.png", dpi=150)
    plt.show()

## Did the model learn?

Trainer logs loss/LR every 50 updates and evaluates both validation splits
after each epoch. Its `global_step` counts completed update attempts; fresh
TraceML step IDs can have a different origin. No checkpoint resume is used
in this experiment.

`train_runtime` includes validation; it differs from the sum of traced step
durations. Compare validation quality alongside runtime. Repeat the paired
pilot in reverse order before making a performance claim.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
results = []
for label, run in runs.items():
    history = pd.DataFrame(run["state"].get("log_history", []))
    if history.empty:
        print(
            f"{label}: final Trainer history is absent; use the saved stdout log."
        )
        continue
    if "loss" in history:
        points = history.dropna(subset=["loss"])
        axes[0].plot(points["step"], points["loss"], label=label)
    if "learning_rate" in history:
        points = history.dropna(subset=["learning_rate"])
        axes[1].plot(points["step"], points["learning_rate"], label=label)
    final = {"run": label, **run["train_metrics"]}
    for split in ("matched", "mismatched"):
        key = f"eval_{split}_accuracy"
        if key in history:
            points = history.dropna(subset=[key])
            axes[2].plot(
                points["epoch"],
                points[key],
                marker="o",
                label=f"{label}/{split}",
            )
        for metric in ("accuracy", "loss"):
            key = f"eval_{split}_{metric}"
            if key in history and history[key].notna().any():
                final[key] = history[key].dropna().iloc[-1]
    results.append(final)
for axis, title, xlabel in zip(
    axes,
    ("Training loss", "Learning rate", "Validation accuracy"),
    ("Trainer update", "Trainer update", "Epoch"),
):
    axis.set(title=title, xlabel=xlabel)
    if axis.lines:
        axis.legend(fontsize=8)
    axis.grid(alpha=0.2)
fig.tight_layout()
for run in runs.values():
    fig.savefig(run["path"] / "analysis/learning.png", dpi=150)
plt.show()
display(pd.DataFrame(results))

## Explore the saved data

`runs[label]["timing"]` contains per-step phase values, including both outer
timing clocks. `runs[label]["memory"]` contains allocator peaks.

List the database tables below. To inspect System, Process or another table,
edit the final query and re-run the cell. The connection remains read-only.
Use explicit `global_rank`, `node_rank`, `gpu_idx` and timestamp columns when
joining data; do not join different ranks or devices just on step number.


In [ ]:
selected = next(iter(runs.values()))
print("Database:", selected["database"])
print("Tables:", selected["tables"])
with closing(
    sqlite3.connect(
        selected["database"].resolve().as_uri() + "?mode=ro", uri=True
    )
) as conn:
    display(pd.read_sql_query("SELECT * FROM step_time_samples LIMIT 5", conn))